In [4]:
%pip install tensorflow

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [1]:
# ============================================================
# PLANT NUTRIENT DEFICIENCY - TINYML CNN
# ============================================================

# ============================================================
# 1. IMPORT LIBRARIES
# ============================================================

import os
import zipfile
import shutil
import tensorflow as tf

from tensorflow.keras import layers, models
from tensorflow.keras.preprocessing import image_dataset_from_directory


# ============================================================
# 2. DATASET PATH
# ============================================================

ZIP_DATASET_PATH = r"C:\Users\ad212\Downloads\archive(5).zip"

EXTRACT_PATH = "extracted_dataset"

IMAGE_SIZE = (96, 96)

BATCH_SIZE = 16

EPOCHS = 30


# ============================================================
# 3. CHECK ZIP FILE
# ============================================================

if not os.path.isfile(ZIP_DATASET_PATH):
    raise FileNotFoundError(
        "ZIP file not found:\n" + ZIP_DATASET_PATH
    )

print("ZIP file found successfully.")


# ============================================================
# 4. EXTRACT ZIP
# ============================================================

if os.path.exists(EXTRACT_PATH):
    shutil.rmtree(EXTRACT_PATH)

os.makedirs(EXTRACT_PATH, exist_ok=True)

print("\nExtracting dataset...")

with zipfile.ZipFile(ZIP_DATASET_PATH, "r") as zip_ref:
    zip_ref.extractall(EXTRACT_PATH)

print("Dataset extracted successfully.")


# ============================================================
# 5. DATASET PATHS
# ============================================================

DATASET_PATH = os.path.join(
    EXTRACT_PATH,
    "Nutrition_dataset"
)

TRAIN_PATH = os.path.join(
    DATASET_PATH,
    "train"
)

TEST_PATH = os.path.join(
    DATASET_PATH,
    "test"
)


print("\n========================================")
print("DATASET PATH")
print("========================================")

print("Dataset :", DATASET_PATH)
print("Train   :", TRAIN_PATH)
print("Test    :", TEST_PATH)


# ============================================================
# 6. CHECK FOLDERS
# ============================================================

if not os.path.isdir(DATASET_PATH):
    raise FileNotFoundError(
        "Nutrition_dataset folder was not found."
    )

if not os.path.isdir(TRAIN_PATH):
    raise FileNotFoundError(
        "train folder was not found."
    )

if not os.path.isdir(TEST_PATH):
    raise FileNotFoundError(
        "test folder was not found."
    )

print("\nTrain and test folders found.")


# ============================================================
# 7. FIND CLASS NAMES AUTOMATICALLY
# ============================================================

CLASS_NAMES = sorted(
    [
        folder
        for folder in os.listdir(TRAIN_PATH)
        if os.path.isdir(
            os.path.join(TRAIN_PATH, folder)
        )
    ]
)

if len(CLASS_NAMES) == 0:
    raise ValueError(
        "No class folders found inside train folder."
    )

NUM_CLASSES = len(CLASS_NAMES)


print("\n========================================")
print("CLASSES")
print("========================================")

for i, name in enumerate(CLASS_NAMES):
    print(i, "->", name)

print("\nNumber of classes:", NUM_CLASSES)


# ============================================================
# 8. CHECK TEST CLASSES
# ============================================================

TEST_CLASSES = sorted(
    [
        folder
        for folder in os.listdir(TEST_PATH)
        if os.path.isdir(
            os.path.join(TEST_PATH, folder)
        )
    ]
)

print("\nTest classes:")

for name in TEST_CLASSES:
    print("->", name)


# ============================================================
# 9. MAKE SURE TRAIN AND TEST CLASSES MATCH
# ============================================================

if CLASS_NAMES != TEST_CLASSES:
    raise ValueError(
        "\nTrain and test class folders do not match.\n\n"
        "Train classes: "
        + str(CLASS_NAMES)
        + "\n\nTest classes: "
        + str(TEST_CLASSES)
    )

print("\nTrain and test classes match.")


# ============================================================
# 10. LOAD TRAIN DATA
# ============================================================

print("\n========================================")
print("LOADING TRAINING DATA")
print("========================================")

train_dataset = image_dataset_from_directory(

    TRAIN_PATH,

    labels="inferred",

    label_mode="int",

    class_names=CLASS_NAMES,

    image_size=IMAGE_SIZE,

    batch_size=BATCH_SIZE,

    shuffle=True,

    seed=123
)


# ============================================================
# 11. LOAD TEST DATA
# ============================================================

print("\n========================================")
print("LOADING TEST DATA")
print("========================================")

test_dataset = image_dataset_from_directory(

    TEST_PATH,

    labels="inferred",

    label_mode="int",

    class_names=CLASS_NAMES,

    image_size=IMAGE_SIZE,

    batch_size=BATCH_SIZE,

    shuffle=False
)


# ============================================================
# 12. OPTIMIZE DATA PIPELINE
# ============================================================

AUTOTUNE = tf.data.AUTOTUNE

train_dataset = train_dataset.prefetch(
    AUTOTUNE
)

test_dataset = test_dataset.prefetch(
    AUTOTUNE
)


# ============================================================
# 13. DATA AUGMENTATION
# ============================================================

data_augmentation = tf.keras.Sequential([

    layers.RandomFlip(
        "horizontal"
    ),

    layers.RandomRotation(
        0.1
    ),

    layers.RandomZoom(
        0.1
    )

])


# ============================================================
# 14. CREATE CNN MODEL
# ============================================================

model = models.Sequential([

    layers.Input(
        shape=(96, 96, 3)
    ),

    data_augmentation,

    layers.Rescaling(
        1.0 / 255
    ),

    layers.Conv2D(
        16,
        (3, 3),
        activation="relu",
        padding="same"
    ),

    layers.MaxPooling2D(),

    layers.Conv2D(
        32,
        (3, 3),
        activation="relu",
        padding="same"
    ),

    layers.MaxPooling2D(),

    layers.Conv2D(
        64,
        (3, 3),
        activation="relu",
        padding="same"
    ),

    layers.MaxPooling2D(),

    layers.GlobalAveragePooling2D(),

    layers.Dense(
        32,
        activation="relu"
    ),

    layers.Dropout(
        0.2
    ),

    layers.Dense(
        NUM_CLASSES,
        activation="softmax"
    )

])


# ============================================================
# 15. COMPILE MODEL
# ============================================================

model.compile(

    optimizer=tf.keras.optimizers.Adam(
        learning_rate=0.001
    ),

    loss="sparse_categorical_crossentropy",

    metrics=["accuracy"]

)


# ============================================================
# 16. DISPLAY MODEL
# ============================================================

print("\n========================================")
print("MODEL ARCHITECTURE")
print("========================================")

model.summary()


# ============================================================
# 17. TRAIN MODEL
# ============================================================

print("\n========================================")
print("STARTING TRAINING")
print("========================================")

history = model.fit(

    train_dataset,

    validation_data=test_dataset,

    epochs=EPOCHS

)


# ============================================================
# 18. EVALUATE MODEL
# ============================================================

print("\n========================================")
print("MODEL EVALUATION")
print("========================================")

loss, accuracy = model.evaluate(
    test_dataset
)

print(
    "\nTest Accuracy: "
    + str(round(accuracy * 100, 2))
    + "%"
)

print(
    "Test Loss: "
    + str(round(loss, 4))
)


# ============================================================
# 19. SAVE KERAS MODEL
# ============================================================

KERAS_FILE = "plant_tinyml_model.keras"

model.save(
    KERAS_FILE
)

print(
    "\nSaved:",
    KERAS_FILE
)


# ============================================================
# 20. FLOAT32 TFLITE
# ============================================================

print("\n========================================")
print("FLOAT32 TFLITE CONVERSION")
print("========================================")

converter = tf.lite.TFLiteConverter.from_keras_model(
    model
)

tflite_float32 = converter.convert()

FLOAT32_FILE = "plant_model_float32.tflite"

with open(
    FLOAT32_FILE,
    "wb"
) as f:

    f.write(
        tflite_float32
    )

print(
    "Saved:",
    FLOAT32_FILE
)


# ============================================================
# 21. REPRESENTATIVE DATASET FOR INT8
# ============================================================

def representative_dataset():

    for images, labels in train_dataset.take(100):

        image = images[0:1]

        yield [
            tf.cast(
                image,
                tf.float32
            )
        ]


# ============================================================
# 22. INT8 TFLITE CONVERSION
# ============================================================

print("\n========================================")
print("INT8 TFLITE CONVERSION")
print("========================================")

converter = tf.lite.TFLiteConverter.from_keras_model(
    model
)

converter.optimizations = [
    tf.lite.Optimize.DEFAULT
]

converter.representative_dataset = (
    representative_dataset
)

converter.target_spec.supported_ops = [
    tf.lite.OpsSet.TFLITE_BUILTINS_INT8
]

converter.inference_input_type = tf.int8

converter.inference_output_type = tf.int8

tflite_int8 = converter.convert()


# ============================================================
# 23. SAVE INT8 MODEL
# ============================================================

INT8_FILE = "plant_model_int8.tflite"

with open(
    INT8_FILE,
    "wb"
) as f:

    f.write(
        tflite_int8
    )

print(
    "Saved:",
    INT8_FILE
)


# ============================================================
# 24. SAVE LABELS
# ============================================================

LABEL_FILE = "labels.txt"

with open(
    LABEL_FILE,
    "w"
) as f:

    for label in CLASS_NAMES:

        f.write(
            label + "\n"
        )

print(
    "Saved:",
    LABEL_FILE
)


# ============================================================
# 25. MODEL SIZE
# ============================================================

model_size_kb = len(
    tflite_int8
) / 1024

print(
    "\nINT8 Model Size:",
    round(model_size_kb, 2),
    "KB"
)


# ============================================================
# 26. FINAL OUTPUT
# ============================================================

print("\n========================================")
print("       TRAINING COMPLETED")
print("========================================")

print("\nClasses:")

for i, label in enumerate(CLASS_NAMES):

    print(
        i,
        "->",
        label
    )

print("\nGenerated files:")

print(
    "1. plant_tinyml_model.keras"
)

print(
    "2. plant_model_float32.tflite"
)

print(
    "3. plant_model_int8.tflite"
)

print(
    "4. labels.txt"
)

print("\nFinal Test Accuracy:")
print(
    round(accuracy * 100, 2),
    "%"
)

print("\n========================================")
print("             DONE")
print("========================================")

ZIP file found successfully.

Extracting dataset...
Dataset extracted successfully.

DATASET PATH
Dataset : extracted_dataset\Nutrition_dataset
Train   : extracted_dataset\Nutrition_dataset\train
Test    : extracted_dataset\Nutrition_dataset\test

Train and test folders found.

CLASSES
0 -> ALL Present
1 -> ALLAB
2 -> KAB
3 -> NAB
4 -> PAB
5 -> ZNAB

Number of classes: 6

Test classes:
-> ALL Present
-> ALLAB
-> KAB
-> NAB
-> PAB
-> ZNAB

Train and test classes match.

LOADING TRAINING DATA
Found 12795 files belonging to 6 classes.

LOADING TEST DATA
Found 4832 files belonging to 6 classes.

MODEL ARCHITECTURE


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ sequential (Sequential)              │ (None, 96, 96, 3)           │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ rescaling (Rescaling)                │ (None, 96, 96, 3)           │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d (Conv2D)                      │ (None, 96, 96, 16)          │             448 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d (MaxPooling2D)         │ (None, 48, 48, 16)          │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_1 (Conv2D)                    │ (None, 48, 48, 32)          │           4,640 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d_1 (MaxPooling2D)       │ (None, 24, 24, 32)          │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_2 (Conv2D)                    │ (None, 24, 24, 64)          │          18,496 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d_2 (MaxPooling2D)       │ (None, 12, 12, 64)          │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ global_average_pooling2d             │ (None, 64)                  │               0 │
│ (GlobalAveragePooling2D)             │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense (Dense)                        │ (None, 32)                  │           2,080 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout (Dropout)                    │ (None, 32)                  │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_1 (Dense)                      │ (None, 6)                   │             198 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 25,862 (101.02 KB)

 Trainable params: 25,862 (101.02 KB)

...

Saved artifact at 'C:\Users\ad212\AppData\Local\Temp\tmp0kygj4qr'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 96, 96, 3), dtype=tf.float32, name='keras_tensor')
Output Type:
  TensorSpec(shape=(None, 6), dtype=tf.float32, name=None)
Captures:
  2691091658512: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2691091659472: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2691091659280: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2691091659856: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2691091659664: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2691091660240: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2691091660048: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2691091660624: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2691091659088: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2691091658704: TensorSpec(shape=(), dtype=tf.resource, name=None)
Saved: pl

INFO:tensorflow:Assets written to: C:\Users\ad212\AppData\Local\Temp\tmpq_op7qd9\assets


Saved artifact at 'C:\Users\ad212\AppData\Local\Temp\tmpq_op7qd9'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 96, 96, 3), dtype=tf.float32, name='keras_tensor')
Output Type:
  TensorSpec(shape=(None, 6), dtype=tf.float32, name=None)
Captures:
  2691091658512: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2691091659472: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2691091659280: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2691091659856: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2691091659664: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2691091660240: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2691091660048: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2691091660624: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2691091659088: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2691091658704: TensorSpec(shape=(), dtype=tf.resource, name=None)


C:\Users\ad212\AppData\Roaming\Python\Python313\site-packages\tensorflow\lite\python\convert.py:846: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(


Saved: plant_model_int8.tflite
Saved: labels.txt

INT8 Model Size: 34.37 KB

       TRAINING COMPLETED

Classes:
0 -> ALL Present
1 -> ALLAB
2 -> KAB
3 -> NAB
4 -> PAB
5 -> ZNAB

Generated files:
1. plant_tinyml_model.keras
2. plant_model_float32.tflite
3. plant_model_int8.tflite
4. labels.txt

Final Test Accuracy:
94.7 %

             DONE
